In [8]:
# ==============================================================
# 04 - MODELO CON PREPROCESADO DE TEXTO (TF-IDF) + RandomForest
# Usa E_PRGM_ACADEMICO (texto) + F_ESTRATOVIVIENDA (categórica)
# ==============================================================

!pip -q install unidecode

# ------------------------------
# 0) dependencias
# ------------------------------
import os, joblib
import numpy as np
import pandas as pd
from unidecode import unidecode

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [9]:
# ------------------------------
# 1) constantes
# ------------------------------
TARGET   = "RENDIMIENTO_GLOBAL"
COL_TXT  = "E_PRGM_ACADEMICO"   # texto
COL_CAT  = "F_ESTRATOVIVIENDA"  # categórica
FEATURES = [COL_TXT, COL_CAT]

In [10]:
# ------------------------------
# 2) utilidades de limpieza (MISMA normalización que en otros .ipynb)
# ------------------------------
def norm(s):
    """Normaliza texto: quita tildes, trim, mayúsculas y colapsa espacios."""
    if pd.isna(s):
        return s
    t = unidecode(str(s)).strip().upper()
    return " ".join(t.split())

def squeeze_1d(X):
    """Convierte (n,1) a (n,) para alimentar TfidfVectorizer."""
    return np.asarray(X).ravel()

In [11]:
# ------------------------------
# 3) cargar y limpiar train
# ------------------------------
assert os.path.exists("train.csv"), "No se encontró train.csv"
df = pd.read_csv("train.csv")

# nos quedamos SOLO con las columnas de interés
df = df[FEATURES + [TARGET]].copy()
# --------------------------------------------------------------
# Eliminar filas con TARGET nulo (NECESARIO para stratify=y)
# --------------------------------------------------------------
missing_target = df[TARGET].isna().sum()
if missing_target > 0:
    print(f"Removiendo {missing_target} filas con {TARGET} = NaN")
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)

# --------------------------------------------------------------
# Normalización EXACTA de columnas de entrada
# --------------------------------------------------------------
df[COL_TXT] = df[COL_TXT].apply(norm)
df[COL_CAT] = df[COL_CAT].apply(norm)

# --------------------------------------------------------------
# Preparar X e y
# --------------------------------------------------------------
X = df[FEATURES]
y = df[TARGET]

# --------------------------------------------------------------
# Split estratificado
# --------------------------------------------------------------
x_tr, x_va, y_tr, y_va = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


Removiendo 1 filas con RENDIMIENTO_GLOBAL = NaN


In [12]:
# ------------------------------
# 4) pipelines con IMPUTACIÓN para evitar NaNs
# ------------------------------
txt_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="")),
    ("squeeze", FunctionTransformer(squeeze_1d, validate=False)),
    ("tfidf",   TfidfVectorizer(min_df=5, ngram_range=(1,2)))
])

# NOTA: no usamos sparse_output para máxima compatibilidad entre versiones.
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer (queda SPARSE)
pre = ColumnTransformer(transformers=[
    ("txt", txt_pipe, [COL_TXT]),
    ("cat", cat_pipe, [COL_CAT]),
])

# Para usar RandomForest (que requiere DENSE), reducimos con SVD (produce denso)
svd = TruncatedSVD(n_components=150, random_state=42)

# Modelo final
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=42
)

pipe = Pipeline(steps=[
    ("pre", pre),      # TF-IDF + OneHot (salida SPARSE)
    ("svd", svd),      # reduce y convierte a DENSE de 150 comps
    ("clf", rf)        # Random Forest
])

In [13]:
# ------------------------------
# 5) entrenamiento y métricas
# ------------------------------
pipe.fit(x_tr, y_tr)
y_hat = pipe.predict(x_va)

acc = accuracy_score(y_va, y_hat)
f1m = f1_score(y_va, y_hat, average="macro")
print(f"Accuracy validación: {acc:.4f}")
print(f"F1-macro validación: {f1m:.4f}")
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_va, y_hat))

Accuracy validación: 0.3752
F1-macro validación: 0.3499

=== Reporte de Clasificación ===
              precision    recall  f1-score   support

        alto       0.44      0.59      0.50      4054
        bajo       0.39      0.56      0.46      4037
  medio-alto       0.29      0.18      0.22      4034
  medio-bajo       0.28      0.17      0.22      4041

    accuracy                           0.38     16166
   macro avg       0.35      0.37      0.35     16166
weighted avg       0.35      0.38      0.35     16166



In [14]:
# ------------------------------
# 6) guardar artefacto
# ------------------------------
os.makedirs("models", exist_ok=True)
out_model = "models/04_pipeline_rf_tfidf.pkl"
joblib.dump(pipe, out_model)
print(f"\nGuardado: {out_model}")


Guardado: models/04_pipeline_rf_tfidf.pkl


In [15]:
# ------------------------------
# 7) función de SUBMISIÓN (formato Kaggle)
#    Aplica la MISMA normalización y usa el pipeline guardado
# ------------------------------
def make_submission_04(
    path_test="test.csv",
    path_sample="submission_example.csv",
    path_model="models/04_pipeline_rf_tfidf.pkl",
    out_path="submission_04_rf.csv",
    id_col="ID"
):
    assert os.path.exists(path_test),   f"No existe {path_test}"
    assert os.path.exists(path_sample), f"No existe {path_sample}"
    assert os.path.exists(path_model),  f"No existe {path_model}"

    test   = pd.read_csv(path_test)
    sample = pd.read_csv(path_sample)

    # Normalización EXACTA de columnas de entrada
    test[COL_TXT] = test[COL_TXT].apply(norm)
    test[COL_CAT] = test[COL_CAT].apply(norm)

    # cargar pipeline entrenado y predecir
    pipe = joblib.load(path_model)
    yhat = pipe.predict(test[[COL_TXT, COL_CAT]])

    # respetar encabezado del sample
    cols_out   = list(sample.columns)
    target_col = cols_out[1]

    sub = pd.DataFrame({
        cols_out[0]: test[id_col] if id_col in test.columns else sample[cols_out[0]],
        target_col:  yhat
    })
    sub.to_csv(out_path, index=False)
    print(f"Archivo de submission creado: {out_path}")
    return sub

# (opcional) generar una vez
_ = make_submission_04()

Archivo de submission creado: submission_04_rf.csv
